## 데이터 수집 자동화

### 다나와 상품정보 스크래핑

### 자동로그인

- 패스워드 숨기기 
- setx 명령어로 시스템 저장방법

#### 1.  setx 명령어로 시스템에 저장방법
- powershell

``` 
> setx DANAWA_PASSWORD "비밀번호"
```
성공: 지정한 값을 저장했습니다.

```
> setx DANAWA_ID "아이디"
```
성공: 지정한 값을 저장했습니다.

- sysdm.cpl 에서 고급>환경변수 저장된 패스워드 삭제

#### 2. keyring 으로 원도우 자격증명 저장소
- setx와 달리 아예 오픈 안됨
- 패키지 설치

In [1]:
!pip install keyring

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import keyring
from getpass import getpass #password 보안을 위해 사용

In [3]:
username = input('다나와 아이디')
password = getpass('password')

#아이디 저장
keyring.set_password('danawa.com', 'user_id', username)
#패스워드 저장
keyring.set_password('danawa.com', username, password)

print('아이디/패스워드 저장완료')

아이디/패스워드 저장완료


In [4]:
user_id = keyring.get_password('danawa.com', 'user_id')
user_id

''

In [5]:
password = keyring.get_password('danawa.com', user_id)
password is not None

True

In [1]:
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [57]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://www.danawa.com/')

wait.until(
    EC.presence_of_element_located((By.ID, 'danawa_footer'))
)


<selenium.webdriver.remote.webelement.WebElement (session="b58ea37c470346a553f83650d418e9d5", element="f.6D672556F3D2C20B28C9305EEF0E83BB.d.7475A250048810320D74BC63A8FF5C75.e.80")>

In [8]:
# 로그인 링크 확인 후 클릭
driver.find_element(By.LINK_TEXT, '로그인').click()

In [9]:
# 로그인하기
id_input = driver.find_element(By.CSS_SELECTOR, 'input[type="text"]')
id_input.send_keys(user_id)

pass_input = driver.find_element(By.CSS_SELECTOR, 'input[type="password"]')
pass_input.send_keys(password)

In [10]:
login_btn = driver.find_element(By.CLASS_NAME, 'btn_login')
login_btn.click()

#### 동적 페이지 스크래핑

- https://prod.danawa.com/list/?cate=11255834&15main_11_02

In [47]:
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

#wait.until(
#   EC.presence_of_element_located((By.TAG_NAME, 'footer'))
#)

wait.until(
    EC.visibility_of_element_located((By.TAG_NAME, 'footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="4d47cdba861396cc687dca0792095905", element="f.A8DB0A2E53B1F8D4CCDA2A9336FE2F31.d.9ACEC5C74C64DE03A2D3D7DC2DC80DD0.e.22071")>

- select 태그 페이지 내에 1개만 존재

In [14]:
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.text == '90개':
        option.click()


wait.until(
    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="c4fb9a5c039eeafdbea826ae237f0e97", element="f.07577C1E05292A6ACA6382EBFBE67F4A.d.D8495AFE3A0FCECAD25B06688FC919C5.e.13513")>

In [15]:
products = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]')
len(products)

90

In [26]:
product = products[2]
product.text

'3\n포유컴퓨터 퍼포먼스PC 47 R5 7500F RTX5060Ti\n7500F / RTX 5060 Ti / (AMD) B650 / OS미포함 / 700W / AMD / 라이젠 7000시리즈 / 라이젠5 / 라파엘 / DDR5 / 16GB / M.2 / 500GB / NVIDIA / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / 어항형 / LED쿨러 / 용도: 게임용\n혜택 최저가\n1,524,820원\n[G마켓] 삼성카드\n/\n무이자 최대 24개월\n기획전\n전설의 시작 붉은사막 벤치 기반 PC 추천\n23.08. 등록\n브랜드로그\n의견 171\n4.9\n리뷰수\n(570)\n관심\n대량구매\n16GB, M.2 500GB\n1,579,200\n원\n5몰\nVS상품비교에 추가\n2위\n16GB, M.2 1TB\n1,689,600\n원\n5몰\nVS상품비교에 추가\n32GB, M.2 500GB\n1,871,990\n원\n5몰\nVS상품비교에 추가\n1위\n32GB, M.2 1TB\n1,959,360\n원\n5몰\nVS상품비교에 추가\n32GB, M.2 2TB\n2,226,650\n원\n5몰\nVS상품비교에 추가\n32GB, M.2 1TB + HDD 2TB\n2,224,320\n원\n5몰\nVS상품비교에 추가'

In [17]:
# 1번 인덱스 값에서 제품명, 제품스펙 2번 인덱스 값에서 가격 가져오기
product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')

[<selenium.webdriver.remote.webelement.WebElement (session="c4fb9a5c039eeafdbea826ae237f0e97", element="f.07577C1E05292A6ACA6382EBFBE67F4A.d.D8495AFE3A0FCECAD25B06688FC919C5.e.13810")>,
 <selenium.webdriver.remote.webelement.WebElement (session="c4fb9a5c039eeafdbea826ae237f0e97", element="f.07577C1E05292A6ACA6382EBFBE67F4A.d.D8495AFE3A0FCECAD25B06688FC919C5.e.13811")>,
 <selenium.webdriver.remote.webelement.WebElement (session="c4fb9a5c039eeafdbea826ae237f0e97", element="f.07577C1E05292A6ACA6382EBFBE67F4A.d.D8495AFE3A0FCECAD25B06688FC919C5.e.13812")>]

In [18]:
computer_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[1]
computer_info.text

'1\n영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080\n9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W / AMD / 라이젠 9000시리즈 / 라이젠7 / 그래니트 릿지 / DDR5 / 32GB / M.2 / 1TB / NVIDIA / 그래픽 메모리: 16GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / 어항형 / 수랭쿨러 / LED쿨러 / 용도: 게임용\n혜택 최저가\n4,379,770원\n[G마켓] 삼성카드\n/\n무이자 최대 24개월\n기획전\n오늘 사는 PC가 가장 저렴한 PC!\n24.12. 등록\n브랜드로그\n4.8\n리뷰수\n(11)\n관심\n대량구매'

In [19]:
# 컴퓨터 이름 출력
computer_info.find_element(By.CSS_SELECTOR, 'div a').text

'영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080'

In [20]:
# 컴퓨터 사양 출력
specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')

'/'.join(spec.text for spec in specs)

'9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W / AMD / 라이젠 9000시리즈 / 라이젠7 / 그래니트 릿지 / DDR5 / 32GB / M.2 / 1TB / NVIDIA / 그래픽 메모리: 16GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / 어항형 / 수랭쿨러 / LED쿨러 / 용도: 게임용'

In [21]:
price_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[2]
price_info.text

'32GB, M.2 1TB\n4,529,810\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.'

In [ ]:
# 가격정보 추출
price_info.find_element(By.CSS_SELECTOR, 'li').text

'32GB, M.2 1TB\n4,529,810\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.'

In [ ]:
# 인덱스로 데이터 가져오면서 필요값만 필터링
# 원이 나오면 반복 종료
price_info.find_elements(By.CSS_SELECTOR, 'li > div span')[0].text

'32GB, M.2 1TB'

In [32]:
spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span')

for span in spans:
    if ('위' in span.text): continue
    if ('원' in span.text): break
    print(span.text)

32GB, M.2 1TB
4,529,810


- 각 부분별 동작확인완료

##### 한페이지 90개 데이터 스크래핑하기

In [58]:
# 다나와 게시판 게이밍PC 페이지로 이동
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    # 모든 요소가 화면에 완전표시될때까지 대기
    EC.visibility_of_all_elements_located((By.TAG_NAME, 'footer'))
)

# 게시물 개수 30개에서 90개로 늘리기
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

# 화면에 90개 데이터를 다 조회할때까지 대기
products = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
)

# list를 담을 수 있는 배열 생성
danawa_prices = []

# 총 게시물 읽어오기
# len(products)
for index, product in enumerate(products):
    # print(index, product.text)
    items = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')
    computer_info = items[1]  # 컴퓨터 명, 사양 정보 들어있는 영역
    price_info = items[2]   # 가격정보 들어있는 영역

    # print(computer_info.text)
    ## Pandas DF에 컬럼으로 사용할 이름들 사용
    ## index, computer_name, computer_spec
    computer_name = computer_info.find_element(By.CSS_SELECTOR, 'div a').text
    specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
    computer_spec = ' / '.join(spec.text for spec in specs)

    # 가격
    spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span') 
    price_spec = ''
    price = ''

    for span in spans:
        if ('위' in span.text): continue  # 순위 제외
        if ('원' in span.text): break    # 첫번째 금액만 받고 종료

        if ('B' in span.text):
            price_spec = span.text
        else:
            price = span.text

    danawa_prices.append({
        'index': index,
        'computer_name': computer_name,
        'computer_spec': computer_spec,
        'price_spec' : price_spec,
        'price' : price
    })
    

In [59]:
len(danawa_prices)

90

In [60]:
df_danawa = pd.DataFrame(danawa_prices)

In [63]:
df_danawa.tail()

,index,computer_name,computer_spec,price_spec,price
85,85,MSI MAG Codex R2 A5NVM I12 U5-225F RTX 5060 Ti...,225F / RTX 5060 Ti / (인텔) H810 / OS미포함 / 600W ...,"16GB, M.2 1TB","1,759,000"
86,86,STCOM STAI-V138,5600 / RTX 5060 / (AMD) A520 / OS미포함 / 600W / ...,"16GB, M.2 512GB","1,210,710"
87,87,영웅컴퓨터 영웅 울트라게이밍 010,7500F / RTX 5070 Ti / (AMD) B650 / OS미포함 / 800...,"32GB, M.2 1TB","3,409,610"
88,88,다나와표준PC 게임용 260114,9600X / RX 9060 / (AMD) B650 / OS미포함 / 850W / ...,"16GB, M.2 1TB","1,921,100"
89,89,포유컴퓨터 퍼포먼스PC 46 R5 7400F RTX5060,7400F / RTX 5060 / (AMD) B650 / OS미포함 / 600W /...,"16GB, M.2 1TB","1,611,840"


##### 페이지 확인

In [68]:
# 
buttons = driver.find_elements(By.CSS_SELECTOR, 'nav > nav div button')
len(buttons)

10

In [77]:
for button in buttons:
    button.click()               # 페이지 이동 → 버튼들이 새로 생성될 수 있음
    page_num = int(button.text)   # 이전 버튼을 다시 읽는 것도 위험

StaleElementReferenceException: Message: stale element reference: stale element not found
  (Session info: chrome=154.0.8037.98); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff7ea689d55+5a55]
	chromedriver!(No symbol) [0x7ff7ea5af540]
	chromedriver!(No symbol) [0x7ff7ea3d45fd]
	chromedriver!(No symbol) [0x7ff7ea3ebf6d]
	chromedriver!(No symbol) [0x7ff7ea3ea9d0]
	chromedriver!(No symbol) [0x7ff7ea3dd355]
	chromedriver!(No symbol) [0x7ff7ea3db13e]
	chromedriver!(No symbol) [0x7ff7ea3df625]
	chromedriver!(No symbol) [0x7ff7ea3df6ef]
	chromedriver!(No symbol) [0x7ff7ea4321a0]
	chromedriver!(No symbol) [0x7ff7ea423791]
	chromedriver!(No symbol) [0x7ff7ea4590ea]
	chromedriver!(No symbol) [0x7ff7ea4230a6]
	chromedriver!(No symbol) [0x7ff7ea47fcd9]
	chromedriver!(No symbol) [0x7ff7ea421a12]
	chromedriver!(No symbol) [0x7ff7ea422823]
	chromedr

In [72]:
button_nextpage = buttons[9]
button_nextpage.click()

In [73]:
# 다 동작했을때 10페이지에 있는지 확인
for button in buttons:
    button.click()

    products = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
)

##### 이전 스크래핑 + 페이지이동 조합
- 통합 소스코드

In [83]:
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    EC.visibility_of_all_elements_located((By.TAG_NAME, 'footer'))
)

options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()


danawa_prices = []
buttons = driver.find_elements(By.CSS_SELECTOR, 'nav > nav div button')

for button in buttons:
    page_num = int(button.text)
    button.click()


    products = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
)


    for index, product in enumerate(products):
        items = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')
        computer_info = items[1]  # 컴퓨터 명, 사양 정보 들어있는 영역
        price_info = items[2]   # 가격정보 들어있는 영역

    
        computer_name = computer_info.find_element(By.CSS_SELECTOR, 'div a').text
        specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
        computer_spec = ' / '.join(spec.text for spec in specs)

        
        spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span') 
        price_spec = ''
        price = ''

        for span in spans:
            if ('위' in span.text): continue  # 순위 제외
            if ('원' in span.text): break    # 첫번째 금액만 받고 종료

            if ('B' in span.text):
                price_spec = span.text
            else:
                price = span.text

        danawa_prices.append({
            'index': ((page_num -1) * 90) + index,
            'computer_name': computer_name,
            'computer_spec': computer_spec,
            'price_spec' : price_spec,
            'price' : price
        })



In [84]:
df_total = pd.DataFrame(danawa_prices)
df_total.info()

<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   index          900 non-null    int64
 1   computer_name  900 non-null    str  
 2   computer_spec  900 non-null    str  
 3   price_spec     900 non-null    str  
 4   price          900 non-null    str  
dtypes: int64(1), str(4)
memory usage: 35.3 KB


In [85]:
df_total.dropna()

,index,computer_name,computer_spec,price_spec,price
0,0,영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080,9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W...,"32GB, M.2 1TB","4,523,480"
1,1,영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060,5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / ...,"16GB, M.2 256GB","1,181,800"
2,2,포유컴퓨터 퍼포먼스PC 47 R5 7500F RTX5060Ti,7500F / RTX 5060 Ti / (AMD) B650 / OS미포함 / 700...,"16GB, M.2 500GB","1,579,200"
3,3,MSI MAG Codex Z2 A9NVR A18 R7-9800X3D RTX 5070...,9800X3D / RTX 5070 Ti / (AMD) B850 / OS미포함 / 8...,"32GB, M.2 1TB","3,943,540"
4,4,영웅컴퓨터 영웅 게이밍울트라 7857 R7 7800X3D RTX5070,7800X3D / RTX 5070 / (AMD) B650 / OS미포함 / 800W...,"32GB, M.2 1TB","2,805,860"
...,...,...,...,...,...
895,895,HP 오멘 35L GT16-0015KL Win11Pro,285K / RTX 5080 / (인텔) Z890 / 윈도우11 프로 / 1000W...,"64GB, M.2 2TB","7,310,000"
896,896,HP 옴니데스크 M03-0038kr Win11Pro,225 / 인텔 그래픽스 / 윈도우11 프로 / 500W / 인텔 / 코어울트라 시...,"16GB, M.2 1TB","1,550,450"
897,897,엔피씨코리아 NEXPC 게이밍프로 G365,9700X / RTX 5060 / (AMD) B650 / OS미포함 / 800W /...,"32GB, M.2 1TB","2,205,720"
898,898,다나와표준PC 게임용 260615,7500X3D / RTX 5070 / (AMD) B850 / OS미포함 / 850W...,"16GB, M.2 1TB","2,799,990"


In [86]:
# 판다스컬럼에 천단위 쉼포(,)는 제거 후 형변환
df_total['price'] = df_total['price'].str.replace(',','')

In [90]:
df= pd.read_csv('../data/danawa_게이밍PC.csv', encoding='utf-8')
df = df.drop(columns=["Unnamed:0"], errors='ignore')

In [91]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   index          900 non-null    int64  
 1   computer_name  900 non-null    str    
 2   computer_spec  900 non-null    str    
 3   price_spec     899 non-null    str    
 4   price          899 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 35.3 KB


In [92]:
df

,index,computer_name,computer_spec,price_spec,price
0,0,영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080,9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W...,"32GB, M.2 1TB",4523480.0
1,1,영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060,5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / ...,"16GB, M.2 256GB",1181800.0
2,2,포유컴퓨터 퍼포먼스PC 47 R5 7500F RTX5060Ti,7500F / RTX 5060 Ti / (AMD) B650 / OS미포함 / 700...,"16GB, M.2 500GB",1579200.0
3,3,MSI MAG Codex Z2 A9NVR A18 R7-9800X3D RTX 5070...,9800X3D / RTX 5070 Ti / (AMD) B850 / OS미포함 / 8...,"32GB, M.2 1TB",3943540.0
4,4,영웅컴퓨터 영웅 게이밍울트라 7857 R7 7800X3D RTX5070,7800X3D / RTX 5070 / (AMD) B650 / OS미포함 / 800W...,"32GB, M.2 1TB",2805860.0
...,...,...,...,...,...
895,895,HP 오멘 35L GT16-0015KL Win11Pro,285K / RTX 5080 / (인텔) Z890 / 윈도우11 프로 / 1000W...,"64GB, M.2 2TB",7310000.0
896,896,HP 옴니데스크 M03-0038kr Win11Pro,225 / 인텔 그래픽스 / 윈도우11 프로 / 500W / 인텔 / 코어울트라 시...,"16GB, M.2 1TB",1550450.0
897,897,엔피씨코리아 NEXPC 게이밍프로 G365,9700X / RTX 5060 / (AMD) B650 / OS미포함 / 800W /...,"32GB, M.2 1TB",2205720.0
898,898,다나와표준PC 게임용 260615,7500X3D / RTX 5070 / (AMD) B850 / OS미포함 / 850W...,"16GB, M.2 1TB",2799990.0


#### DB 저장

- SQLAlchemy 패키지 사용, SQL 쿼리문 없이 CRUD 가능
- `pip install SQLAlchemy`

In [94]:
!pip install SQLAlchemy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ------------------------- -------------- 1.6/2.4 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 5.3 MB/s  0:00:00


In [ ]:
from datetime import datetime
from getpass import getpass

#SQLAlchemy
import psycopg
from sqlalchemy import create_engine, URL, Text, BigInteger, Integer

In [96]:
df.head()

,index,computer_name,computer_spec,price_spec,price
0,0,영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080,9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W...,"32GB, M.2 1TB",4523480.0
1,1,영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060,5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / ...,"16GB, M.2 256GB",1181800.0
2,2,포유컴퓨터 퍼포먼스PC 47 R5 7500F RTX5060Ti,7500F / RTX 5060 Ti / (AMD) B650 / OS미포함 / 700...,"16GB, M.2 500GB",1579200.0
3,3,MSI MAG Codex Z2 A9NVR A18 R7-9800X3D RTX 5070...,9800X3D / RTX 5070 Ti / (AMD) B850 / OS미포함 / 8...,"32GB, M.2 1TB",3943540.0
4,4,영웅컴퓨터 영웅 게이밍울트라 7857 R7 7800X3D RTX5070,7800X3D / RTX 5070 / (AMD) B650 / OS미포함 / 800W...,"32GB, M.2 1TB",2805860.0


In [ ]:
# 저장할 테이블명 - danawa_gaming_pc_yymmdd
table_name = 'danawa_gaming_pc_' + datetime.now().strftime('%y%m%d')
print(table_name)

danawa_gaming_pc_261006


In [105]:
# PostgreSQL 연결정보
db_url = URL.create(
    'postgresql+psycopg', 
    username='postgres',
    password='123456',
    host='127.0.0.1',
    port=5432,
    database='postgres'
)

In [106]:
engine = create_engine(db_url)

In [107]:
df.to_sql(
    name=table_name,
    con=engine,
    if_exists='fail',
    index = "False",
    dtype={
        'index' : BigInteger(),
        'computer_name' : Text(),
        'computer_spec' : Text(),
        'price_spec' : Text(),
        'price' : Integer()
    }
)

engine.dispose() 

print(f'{len(df)})건을 저장하였습니다.')

900)건을 저장하였습니다.
